# 1. Processing raw data with Raphael

Raw Neuropixels recordings → **LFP and theta phase** (`lfp` stage) and
→ **sorted, curated units** (`kilosort4` stage), written into Raphael's processed tree.

Every step is a Raphael **job bundle**: a folder that records the inputs (with checksums),
the settings and the environment, so any result can be traced back. The same bundle runs here,
on a GPU workstation or on the cluster.

This notebook uses a small **simulated** recording, so it runs on any laptop in a minute or two.
For your own data, point `DATA_ROOT` and `INPUTS` at your recordings.

In [ ]:
import os
from pathlib import Path

# Run from the repository root, wherever the notebook was opened from.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)

import matplotlib.pyplot as plt
import numpy as np

## Raw data

Two short "sessions" from one simulated Neuropixels 2.0 day (64 channels, 30 s each),
saved the way spikeinterface saves a recording. Real SpikeGLX or Open Ephys folders work the same.

In [ ]:
import spikeinterface.full as si

from starter_project.demo import make_raw_day

DATA_ROOT = Path('data')                              # inputs are given relative to this
folders = make_raw_day(DATA_ROOT / 'raw' / 'demo')
INPUTS = [str(f.relative_to(DATA_ROOT)) for f in folders]

recording = si.load(folders[0])
print(INPUTS)
print(f'{recording.get_num_channels()} channels, {recording.get_total_duration():.0f} s '
      f'at {recording.get_sampling_frequency():.0f} Hz')

In [ ]:
# 50 ms of raw signal on 8 channels
traces = recording.get_traces(start_frame=0, end_frame=1500, channel_ids=recording.channel_ids[:8])
t = np.arange(len(traces)) / recording.get_sampling_frequency() * 1000
plt.figure(figsize=(8, 3))
plt.plot(t, traces + np.arange(8) * 100, lw=0.5, color='k')
plt.xlabel('ms'); plt.yticks([]); plt.title('raw signal, 8 channels');

## LFP: make a job, then run it

`jobs.make` checks the parameters and records the inputs. It does no compute, so on the
cluster you would make the bundle here and run it there (see the last section).
`configs/ephys.yaml` is this project's copy of Raphael's settings; the job keeps a snapshot of it.

In [ ]:
from raphael.cluster import jobs

params = {
    'mouse': '1', 'day': '1',
    'session_type': ['OF1', 'OF2'],                        # one per input
    'session_start': ['2026-01-01T10:00', '2026-01-01T10:30'],
}
lfp_job = jobs.make(Path('jobs') / 'lfp-demo', 'lfp', INPUTS, DATA_ROOT, params=params,
                    resources={'gpu': False}, configs=['configs/ephys.yaml'],
                    hash_inputs=False) if not Path('jobs/lfp-demo').exists() else Path('jobs/lfp-demo')
print((lfp_job / 'job.yaml').read_text()[:800])

In [ ]:
if jobs.read_status(lfp_job)['state'] != 'DONE':
    jobs.run(lfp_job, DATA_ROOT)
print(jobs.read_status(lfp_job))
for f in sorted((lfp_job / 'outputs').rglob('*.nwb')):
    print(f.relative_to(lfp_job / 'outputs'))

## Look at the output

Each session gets an `_lfp.nwb`: the LFP (1 kHz, 8 contacts averaged per channel) and its theta phase.

In [ ]:
import pynwb
from raphael.ephys.layout import Session

session = Session('1', '1', 'OF1')
with pynwb.NWBHDF5IO(str(session.lfp(lfp_job / 'outputs')), 'r') as io:
    nwb = io.read()
    lfp = nwb.processing['ecephys']['LFP']['lfp']
    theta = nwb.processing['ecephys']['Processed']['theta']
    rate = lfp.rate
    x = lfp.data[:2000, 0] * 1.0          # µV, first 2 s of channel 0
    phase = theta.data[:2000, 0]

t = np.arange(len(x)) / rate
fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
axes[0].plot(t, x, lw=0.6); axes[0].set_ylabel('LFP (µV)')
axes[1].plot(t, phase, lw=0.6); axes[1].set_ylabel('theta phase (rad)'); axes[1].set_xlabel('s');

## Spike sorting (Kilosort 4)

Needs the `gpu` extra: `uv sync --extra gpu`. On a laptop without an NVIDIA GPU it runs on the CPU
(about 2 minutes for this demo); real recordings belong on the cluster.
Set `RUN_SORTING = True` to run it.

In [ ]:
RUN_SORTING = False

if RUN_SORTING:
    sort_job = Path('jobs') / 'kilosort4-demo'
    if not sort_job.exists():
        jobs.make(sort_job, 'kilosort4', INPUTS, DATA_ROOT, configs=['configs/ephys.yaml'],
                  params={**params,
                          # simulated channels look "bad" to a detector built for real probes
                          'preprocessing': {'remove_bad_channels': False}},
                  resources={'gpu': False}, hash_inputs=False)
    if jobs.read_status(sort_job)['state'] != 'DONE':
        jobs.run(sort_job, DATA_ROOT)

    from raphael import population
    pop = population.load(session.folder(sort_job / 'outputs')
                          / f'{session.stem}_srt-kilosort4').accepted()
    print(len(pop.ids), 'accepted units')
    print(pop.table[['unit', 'firing_rate', 'bombcell_label']].head())

## On the cluster

Make the bundle here, copy it and the data across, and submit it; Raphael renders the SLURM script
for you (set your login once with `uv run raphael-jobs cluster-user <MUW-ID>`):

```bash
uv run raphael-jobs make jobs/sort-M25-D18 --stage kilosort4 --data-root /data \
    --input raw/M25_D18_OF1 --param mouse=25 --param day=18 \
    --param session_type=OF1 --param session_start=2026-11-03T14:05 \
    --config configs/ephys.yaml
rsync -a jobs/sort-M25-D18 <MUW-ID>@s0-l02.hpc.meduniwien.ac.at:jobs/
ssh <MUW-ID>@s0-l02.hpc.meduniwien.ac.at sbatch jobs/sort-M25-D18/submit.slurm
```

Full walkthrough: Raphael's `docs/msc_cluster.md`.